# Case Study: Fraud & Abuse Detection

Trust & safety ML is characterized by extreme class imbalance, adversarial data, and severe asymmetric costs. This note covers the full design of a fraud/abuse detection system in interview transcript format.

## What Interviewers Test
- Extreme class imbalance handling (cost-sensitive learning, oversampling)
- Adversarial adaptation: why fraud models degrade fast
- Precision-at-review-queue vs recall tradeoffs
- Human-in-the-loop review queues
- Label delay problem for chargebacks
- Rules vs ML layering

## Transcript: Design a Payment Fraud Detection System

**Interviewer:** Design ML for detecting payment fraud at a fintech company. ~100M transactions/day, ~0.5% fraud rate.

**You:** Key clarifications:
1. What is the cost asymmetry? (false negative = lost transaction value; false positive = customer friction)
2. Is there a human review queue? What's its capacity?
3. What's the latency requirement? (real-time before authorization, or post-hoc?)

**Interviewer:** Block in real-time (<100ms). Review queue capacity: 10K transactions/day. FN cost ≈ 100× FP cost.

---

**Scale:** 100M tx/day ≈ 1,160 QPS. 0.5% fraud = 500K fraudulent tx/day. Review queue: 10K/day = 0.01% of transactions.

**Metrics:**
- Primary: Precision@10K (of the 10K we send to review, how many are actual fraud)
- Secondary: Recall at precision threshold (how much fraud we catch)
- Guardrail: False positive rate for good customers, latency p99 < 100ms


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, roc_auc_score
from sklearn.utils import resample
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
np.random.seed(42)

# --- Simulate extreme imbalance ---
n_neg, n_pos = 50000, 250   # ~0.5% fraud
X_neg = np.random.randn(n_neg, 10)
X_pos = np.random.randn(n_pos, 10) + np.array([1,1,1,1,1,-1,-1,-1,-1,-1])
X = np.vstack([X_neg, X_pos])
y = np.array([0]*n_neg + [1]*n_pos)

# --- Cost-matrix approach ---
def cost_matrix_predict(scores, y_true, cost_fn=1, cost_fp=0.01, thresholds=None):
    """Choose threshold that minimizes expected cost."""
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 100)
    costs = []
    for t in thresholds:
        y_pred = (scores >= t).astype(int)
        fn = ((y_pred == 0) & (y_true == 1)).sum()
        fp = ((y_pred == 1) & (y_true == 0)).sum()
        costs.append(fn * cost_fn + fp * cost_fp)
    best_t = thresholds[np.argmin(costs)]
    return best_t, min(costs)

# --- Oversampling (SMOTE-like) ---
X_pos_up = resample(X_pos, n_samples=n_neg, replace=True, random_state=42)
X_bal = np.vstack([X_neg, X_pos_up])
y_bal = np.array([0]*n_neg + [1]*n_neg)

# Train and compare
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_tr_bal, _, y_tr_bal, _ = train_test_split(X_bal, y_bal, test_size=0.2, random_state=42)

lr_imb = LogisticRegression(max_iter=500).fit(X_tr, y_tr)
lr_bal = LogisticRegression(max_iter=500).fit(X_tr_bal, y_tr_bal)

scores_imb = lr_imb.predict_proba(X_te)[:,1]
scores_bal = lr_bal.predict_proba(X_te)[:,1]

print(f"AUC (imbalanced training): {roc_auc_score(y_te, scores_imb):.4f}")
print(f"AUC (oversampled training): {roc_auc_score(y_te, scores_bal):.4f}")

# Precision@K (review queue simulation)
K = 50  # top-50 most suspicious
top_k_imb = y_te[np.argsort(-scores_imb)[:K]]
top_k_bal = y_te[np.argsort(-scores_bal)[:K]]
print(f"Precision@{K} (imbalanced): {top_k_imb.mean():.3f}")
print(f"Precision@{K} (oversampled): {top_k_bal.mean():.3f}")


## Adversarial Adaptation

Fraud patterns evolve in response to detection:
- Fraudsters reverse-engineer the model through probing attacks
- New fraud patterns emerge after a model blocks old ones
- Account takeover patterns differ from synthetic identity fraud

**Why models decay fast:**
- Training data only covers known fraud patterns
- Distribution shift is adversarial (intentional), not natural
- New fraud typologies emerge weekly

**Mitigations:**
- Short retraining cadence (daily or continuous)
- Rules layer that adapts faster than ML (velocity checks, impossible geography)
- Adversarial training: add generated adversarial examples to training data
- Separate models per fraud typology


## Rules vs ML Layering

```
Transaction arrives
  ↓ [Hard rules — instant, deterministic]
    - Impossible geography (TX from two countries in 1 hour)
    - Known blacklisted cards/IPs
    - Velocity checks (5+ transactions in 1 minute)
  ↓ [ML model — probabilistic score]
    - Real-time features (session behavior, device fingerprint)
    - Historical features (merchant risk, user risk)
  ↓ [Human review queue — top K by score]
    - Analysts review borderline cases
    - Their decisions feed back to training

Allow / Block / Hold-for-review
```

**Label delay problem:** Chargebacks (true fraud labels) arrive 30–90 days after the transaction. Mitigate with:
- Proxy labels: issuer decline, customer dispute filed (arrives in days)
- Semi-supervised approaches: use unlabeled transactions with uncertainty sampling
- Evaluation on held-out chargeback-labeled set, not near-real-time labels


## Common Interview Questions

**Q: How do you handle 0.5% fraud rate in training data?**
Multiple approaches: (1) oversampling positives (SMOTE or simple oversampling), (2) undersampling negatives, (3) class-weighted loss (weight positives by ~200×), (4) cost-sensitive learning to explicitly encode FN vs FP asymmetry. In practice, class weighting + cost-sensitive threshold selection at inference is standard.

**Q: Why does a fraud model with high AUC still miss new fraud typologies?**
AUC measures ranking over the test distribution, which was drawn from the same distribution as training. New typologies are out-of-distribution — the model has no signal for them. AUC can be high on known fraud while completely missing new patterns. Solution: monitor false-negative rate on analyst-identified new fraud types; add new typology labels as they emerge.

**Q: What is the label delay problem and how do you handle it?**
Credit card chargebacks — the definitive fraud label — arrive 30–90 days after the transaction. Using real-time training on charge-back labels would introduce severe selection bias (you'd train only on fraud that was reported, not all fraud). Use proxy labels (issuer declines, customer disputes) for faster feedback, and maintain a holdout set with delayed true labels for unbiased offline evaluation.

**Q: What goes in the human review queue?**
High-confidence fraud (block automatically) and clearly legitimate transactions are handled automatically. The review queue receives borderline cases — those with model scores near the operating threshold where the expected cost of automated decision is highest. Analysts' decisions on reviewed cases should feed back into training as additional labeled data.

## Key Takeaways
- Fraud detection metric: Precision@K (review queue) not accuracy — the queue size is the constraint
- Class imbalance: class weights + cost-sensitive threshold selection; oversampling for severe cases
- Adversarial: fraud patterns evolve intentionally; retrain daily, layer rules for fast adaptation
- Rules + ML + human review: three-tier stack with complementary strengths
- Label delay: chargebacks arrive 30–90 days late; use proxy labels for fast feedback
- Hard rules handle known patterns instantly; ML handles probabilistic risk; humans handle edge cases